In [ ]:
import pandas as pd
from eyened_orm.db import Database
from tqdm.notebook import tqdm
from pathlib import Path

## Biomarker post-processing and normalisation

This is a sample notebook showing one way to post-process VascX outputs by:
- Adding derived measurements like the Artery-vein ratio, which is calculated from the CRE biomarkers.
- While several vascx biomarkers such as tortuosity are unit-less, others such as vessel calibers or CREs measure distances. VascX, by default produces distance measurements in pixels and leaves the potential conversion of such measurements up to the user. Here, we optionally normalise them by dividing by the distance between optic disc and fovea (also in pixels) to obtain a unit-less measurement relative to the anatomy. Note that this normalisation may add noise to the measurements. We do not recommend it without validation on the user's specific dataset.
- Some biomarkers such as vascular density are optionally re-scaled to a range more convenient for display.

Important: may need to be adapted for feature sets other than `full_v3`

In [ ]:
FEAT_PATH = Path("../samples/fundus/biomarkers.csv")
OUTPUT_PATH = Path("../samples/fundus/biomarkers_normalized.csv")
df = pd.read_csv(FEAT_PATH, index_col=0)

In [ ]:
cre_features = [col for col in df.columns if '_cre_' in col]
diam_features = [col for col in df.columns if '_diam_' in col]
vd_features = [col for col in df.columns if 'vd_' in col]
rest_cols = sorted(list(set(df.columns) - set(cre_features) - set(diam_features)))

In [ ]:
# # OPTIONAL: normalize features in pixes by dividing by the optic disc to fovea distance
# for col in cre_features + diam_features:
#     df[col] = df[col] / df['disc_fovea_distance_retina']

In [ ]:
print('Caliber biomarkers (OD-fovea-normalised):')
for col in diam_features:
    print(f'- {col}')
print('Central retinal equivalents: (OD-fovea-normalised)')
for col in cre_features:
    print(f'- {col}')
print('Remaining features (no additional normalisation): ')
for col in rest_cols:
    print(f'- {col}')

In [ ]:
# add the artery-vein ratios
for col in ['temporal_cre', 'temporal_cre_hf_superior', 'temporal_cre_hf_inferior', 'nasal_cre', 'full_cre']:
    df[f'avr_{col}'] = df[f'{col}_arteries'] / df[f'{col}_veins']

## Re-scale features (optional)

CREs and diameters are very small after division. Vascular densities can also be converted to percentages.

In [ ]:
for col in cre_features + diam_features + vd_features:
    df[col] = df[col] * 100

In [ ]:
df.to_csv(OUTPUT_PATH)